# FeMoCo resource estimates using THC

**Mahnoor and Stella — PlexTech.** Claim: `femoco-lee-2021`.

We check Lee et al.'s logical counts with OpenFermion and reconstruct their physical
resource model with a custom QDK ISA. This counts resources; it does not simulate molecular energy.

Sources: [paper](https://arxiv.org/pdf/2011.03494v3), OpenFermion's THC costing and
`spacetime.py`, and the [factory formulas cited by the paper](https://arxiv.org/src/1905.09749v3/anc/estimate_costs.py).

The imports below load the tools and check package versions.

In [1]:
import json
import math
import platform
from datetime import datetime, timezone
from importlib import metadata
from pathlib import Path

import pandas as pd
from IPython.display import display
from openfermion.resource_estimates.thc import compute_cost

assert platform.python_version_tuple()[:2] == ("3", "10"), "Use the Python 3.10 environment."
assert metadata.version("openfermion") == "1.7.1"
CLAIM_ID = "femoco-lee-2021"
REPRODUCTION_ID = "openfermion-thc-layout-2026-11"
print({name: metadata.version(name) for name in ("openfermion", "qdk", "qdk-chemistry")})

{'openfermion': '1.7.1', 'qdk': '1.33.1', 'qdk-chemistry': '2.3.0'}


## Paper inputs

THC compresses the molecular interaction data. We use the paper's rank `M`, normalization
`lam`, and bit precisions (`chi`, `beta`). OpenFermion expects spin orbitals, twice the
spatial-orbital count. We include Reiher and both requested/recommended Li ranks.

Phase estimation gets 0.001 Ha of the 0.0016 Ha energy tolerance (p. 14).
`PAPER_TARGETS` contains comparison values, not calculated results.

In [2]:
PHASE_ERROR_HA = 0.001  # p. 14, Eq. (25)
INITIAL_STEP_GUESS = 20000  # OpenFermion reference-check procedure
CASES = {
    "reiher-m350": dict(model="Reiher", orbitals=54, spin_orbitals=108, electrons=54,
                        M=350, lam=306.3, chi=10, beta=16),
    "li-m350": dict(model="Li", orbitals=76, spin_orbitals=152, electrons=113,
                    M=350, lam=1279.0, chi=10, beta=20),
    "li-m450": dict(model="Li", orbitals=76, spin_orbitals=152, electrons=113,
                    M=450, lam=1201.5, chi=10, beta=20),
}
PAPER_TARGETS = {
    "reiher-m350": dict(num_qubits=2142, toffoli_count=5.3e9, locator="Table IV, p. 24"),
    "li-m350": dict(num_qubits=2194, toffoli_count=3.2e10, locator="Table V, p. 24, M=350"),
    "li-m450": dict(num_qubits=2196, toffoli_count=3.2e10, locator="Table V, p. 24, M=450"),
}

## Logical qubits and Toffoli gates

`cost_case` follows OpenFermion's two-pass costing procedure, calculates the number of
walk repetitions, and compares the totals with Tables IV/V. A ratio near one means
agreement. Gate counts that this method does not calculate are omitted, not set to zero.

In [3]:
def cost_case(params):
    args = dict(n=params["spin_orbitals"], lam=params["lam"], dE=PHASE_ERROR_HA,
                chi=params["chi"], beta=params["beta"], M=params["M"])
    first = compute_cost(**args, stps=INITIAL_STEP_GUESS)
    step, total, qubits = map(int, compute_cost(**args, stps=int(first[0])))
    walks = math.ceil(math.pi * params["lam"] / (2 * PHASE_ERROR_HA))
    assert total == step * walks
    return dict(step_toffolis=step, walk_steps=walks, num_qubits=qubits,
                toffoli_count=total, ccz_count=total, ccix_count=0,
                t_count=None, rotation_count=None, rotation_depth=None,
                measurement_count=None)

counts = {key: cost_case(value) for key, value in CASES.items()}
rows = []
for key, result in counts.items():
    target = PAPER_TARGETS[key]
    rows.append(dict(case=key, logical_qubits=result["num_qubits"],
                     paper_qubits=target["num_qubits"], toffolis=result["toffoli_count"],
                     paper_toffolis=target["toffoli_count"],
                     qubit_ratio=result["num_qubits"] / target["num_qubits"],
                     toffoli_ratio=result["toffoli_count"] / target["toffoli_count"],
                     locator=target["locator"]))
display(pd.DataFrame(rows))

,case,logical_qubits,paper_qubits,toffolis,paper_toffolis,qubit_ratio,toffoli_ratio,locator
0,reiher-m350,2142,2142,5250145120,5.300000e+09,1.0,0.990593,"Table IV, p. 24"
1,li-m350,2194,2194,31982051031,3.200000e+10,1.0,0.999439,"Table V, p. 24, M=350"
2,li-m450,2196,2196,31938980976,3.200000e+10,1.0,0.998093,"Table V, p. 24, M=450"


## Layout schedule

`algorithm="half"` reconstructs Fig. 11's half-angle-loading stages. Each stage has a
Toffoli count and live-qubit count. We calculate the loop total and repetitions rather
than entering the paper's rounded answers. This layout differs from the table implementation.

In [4]:
import numpy as np
from openfermion.resource_estimates.thc.spacetime import qubit_vs_toffoli

stage_toffolis, stage_qubits, stage_labels, _ = qubit_vs_toffoli(
    lam=CASES["reiher-m350"]["lam"], dE=PHASE_ERROR_HA,
    eps=PHASE_ERROR_HA / (10 * CASES["reiher-m350"]["lam"]),
    n=108, chi=10, beta=16, M=350, algorithm="half")
assert np.all(stage_toffolis == np.floor(stage_toffolis))
assert np.all(stage_qubits == np.floor(stage_qubits))
stage_toffolis = [int(n) for n in stage_toffolis]
stage_qubits = [int(n) for n in stage_qubits]
WALKS = counts["reiher-m350"]["walk_steps"]
STAGES = [dict(index=i, label=str(label), toffolis=t, live_data_qubits=q)
          for i, (label, t, q) in enumerate(zip(stage_labels, stage_toffolis, stage_qubits))]
layout_logical = dict(walk_steps=WALKS, toffolis_per_walk=sum(stage_toffolis),
                      toffoli_count=sum(stage_toffolis) * WALKS,
                      peak_data_qubits=max(stage_qubits), stage_count=len(STAGES))
display(pd.DataFrame(STAGES))
print(layout_logical)

,index,label,toffolis,live_data_qubits
0,0,small element,95,210
1,1,small element,89,267
2,2,preparation QROM,4293,685
3,3,small element,10,233
4,4,small element,18,223
...,...,...,...,...
83,83,inverse QROM,497,457
84,84,small element,89,267
85,85,small element,95,212
86,86,small element,50,225


{'walk_steps': 481135, 'toffolis_per_walk': 13881, 'toffoli_count': 6678634935, 'peak_data_qubits': 685, 'stage_count': 88}


## Factories, timing and errors

These functions implement the cited factory formulas. They calculate factory dimensions,
CCZ production rate and error. A tick is the slower of factory supply and classical reaction time.

We use the paper's floorplan and code distances. Storage error covers nonfactory patches;
factory error is counted separately to avoid double-counting. Both the main and optimistic
hardware cases must fit the 1% hardware failure budget.

In [5]:
def logical_cell_error(distance, physical_error_rate):
    return 0.1 * (100 * physical_error_rate) ** ((distance + 1) / 2)

def factory_model(physical_error_rate, first_distance, second_distance):
    ratio = first_distance / second_distance
    t_width, t_height, t_depth = 8 * ratio, 4 * ratio, 5.75 * ratio
    first_factories = math.ceil(8 * t_depth / 5)
    width = math.ceil(2 * t_width + 3 + 2 * ratio)
    height = math.ceil(max(6, t_height * math.ceil(first_factories / 2)))
    depth_beats = max(5, t_depth)
    p0 = physical_error_rate + 100 * logical_cell_error(first_distance // 2, physical_error_rate)
    p1 = 35 * p0 ** 3 + 1100 * logical_cell_error(first_distance, physical_error_rate)
    p_ccz = 28 * p1 ** 2 + 1000 * logical_cell_error(second_distance, physical_error_rate)
    return dict(width_patches=width, height_patches=height, depth_beats=depth_beats,
                first_level_factories=first_factories, ccz_error=p_ccz)

HARDWARE_CASES = [
    dict(scenario_id="lee-ccz-layout-main", p=1e-3, data_distance=31,
         first_distance=19, second_distance=31),
    dict(scenario_id="lee-ccz-layout-optimistic", p=1e-4, data_distance=15,
         first_distance=9, second_distance=15),
]
FLOORPLAN = dict(width_patches=53, height_patches=36)  # Fig. 10; retained at optimistic distance, p. 31
NUM_FACTORIES = 4
CYCLE_NS = 1000
REACTION_NS = 10000
FAILURE_BUDGET = 0.01

def hardware_parameters(case):
    f = factory_model(case["p"], case["first_distance"], case["second_distance"])
    patches = FLOORPLAN["width_patches"] * FLOORPLAN["height_patches"]
    factory_patches = NUM_FACTORIES * f["width_patches"] * f["height_patches"]
    nonfactory = patches - factory_patches
    period_ns = f["depth_beats"] * case["second_distance"] * CYCLE_NS
    tick_ns = math.ceil(max(REACTION_NS, period_ns / NUM_FACTORIES))
    beat_ns = case["data_distance"] * CYCLE_NS
    assert tick_ns >= beat_ns, "This schedule needs explicit beat-limited stalls in this regime."
    assert max(stage_qubits) < nonfactory, "Floorplan cannot accommodate live data."
    return dict(factory=f, floorplan_patches=patches, factory_patches=factory_patches,
                nonfactory_patches=nonfactory, physical_qubits_per_patch=2*(case["data_distance"]+1)**2,
                factory_period_ns=period_ns, aggregate_rate_per_second=NUM_FACTORIES*1e9/period_ns,
                tick_ns=tick_ns, beat_ns=beat_ns)

print(pd.DataFrame([dict(scenario=c["scenario_id"], **hardware_parameters(c)["factory"],
                         rate_hz=hardware_parameters(c)["aggregate_rate_per_second"])
                    for c in HARDWARE_CASES]).to_string(index=False))

                 scenario  width_patches  height_patches  depth_beats  first_level_factories    ccz_error      rate_hz
      lee-ccz-layout-main             15               8            5                      6 1.028489e-13 25806.451613
lee-ccz-layout-optimistic             14               8            5                      6 1.356268e-14 53333.333333


## Physical resources with QDK

`LeeLayoutArchitecture` gives QDK a duration and failure bound for each resource stage.
`estimate_layout` repeats those stages, reserves the floorplan, and returns qubits and runtime.
The assertions cross-check QDK's totals against the formulas.

This is a custom resource-level model, not QDK Chemistry compiling every gate. It accepts
the published geometry/distances and omits warmup, state preparation and detailed routing.

In [6]:
from dataclasses import dataclass
from qdk.qre import Architecture, Trace, LOGICAL, linear_function
from qdk.qre.instruction_ids import LATTICE_SURGERY

@dataclass
class LeeLayoutArchitecture(Architecture):
    case: dict

    def provided_isa(self, ctx):
        hw = hardware_parameters(self.case)
        # Supplies the physical patch size. Storage errors are included below,
        # so this sizing instruction must not also charge them.
        nodes = [ctx.add_instruction(
            LATTICE_SURGERY, encoding=LOGICAL, arity=None,
            time=hw["beat_ns"], space=linear_function(hw["physical_qubits_per_patch"]),
            error_rate=0.0)]
        for stage in STAGES:
            duration_ns = stage["toffolis"] * hw["tick_ns"]
            factory_error = stage["toffolis"] * hw["factory"]["ccz_error"]
            storage_error = (hw["nonfactory_patches"] * duration_ns / CYCLE_NS
                             * logical_cell_error(self.case["data_distance"], self.case["p"]))
            nodes.append(ctx.add_instruction(
                50000 + stage["index"], encoding=LOGICAL, arity=1,
                time=duration_ns, space=hw["physical_qubits_per_patch"],
                error_rate=factory_error + storage_error))
        return ctx.make_isa(*nodes)

def estimate_layout(case):
    hw = hardware_parameters(case)
    ctx = LeeLayoutArchitecture(case).context()
    trace = Trace(hw["floorplan_patches"])
    loop = trace.add_block(repetitions=WALKS)
    for stage in STAGES:
        # Each macro shares a dependency so the algorithm stages are sequential.
        loop.add_operation(50000 + stage["index"], [0])
    result = trace.estimate(ctx._isa, max_error=FAILURE_BUDGET)
    if result is None:
        raise RuntimeError("No estimate within the fixed error budget; investigate without tuning inputs.")
    manual_runtime_ns = layout_logical["toffoli_count"] * hw["tick_ns"]
    assert result.runtime == manual_runtime_ns
    assert result.qubits == hw["floorplan_patches"] * hw["physical_qubits_per_patch"]
    manual_factory_error = layout_logical["toffoli_count"] * hw["factory"]["ccz_error"]
    manual_storage_error = (hw["nonfactory_patches"] * manual_runtime_ns / CYCLE_NS
                            * logical_cell_error(case["data_distance"], case["p"]))
    assert math.isclose(result.error, manual_factory_error + manual_storage_error, rel_tol=1e-10)
    return {
        "scenario_id": case["scenario_id"],
        "architecture": {"physical_error_rate": case["p"], "reaction_time_ns": REACTION_NS},
        "code": {"family": "surface", "distance": case["data_distance"],
                 "distance_source": "imposed", "cycle_time_ns": CYCLE_NS},
        "factories": f"{NUM_FACTORIES} CCZ factories; distances {case['first_distance']}/{case['second_distance']}",
        "synthesis": "OpenFermion half-angle THC stages; custom Lee-layout QDK ISA; phase-gradient costs included",
        "physical_qubits": int(result.qubits),
        "compute_qubits": hw["nonfactory_patches"] * hw["physical_qubits_per_patch"],
        "factory_qubits": hw["factory_patches"] * hw["physical_qubits_per_patch"],
        "memory_qubits": 0, "runtime_seconds": result.runtime / 1e9,
        "error_budget": FAILURE_BUDGET, "achieved_error": float(result.error),
    }, dict(parameters=case, derived=hw, factory_error_bound=manual_factory_error,
            nonfactory_error_bound=manual_storage_error)

physical_scenarios = []
hardware_diagnostics = {}
for case in HARDWARE_CASES:
    scenario, diagnostic = estimate_layout(case)
    physical_scenarios.append(scenario)
    hardware_diagnostics[case["scenario_id"]] = diagnostic
display(pd.DataFrame([dict(scenario=x["scenario_id"], physical_qubits=x["physical_qubits"],
                           days=x["runtime_seconds"]/86400, failure_bound=x["achieved_error"])
                      for x in physical_scenarios]))

,scenario,physical_qubits,days,failure_bound
0,lee-ccz-layout-main,3907584,2.995337,0.004383
1,lee-ccz-layout-optimistic,976896,1.449357,0.001919


## Reading the results

Compare table logical counts and layout physical estimates separately. Close agreement
checks the published resource model under its assumptions; it does not independently
validate chemical accuracy, layout feasibility or a real quantum device.

## Save results

The final cell writes `results.json` from the calculated values. It keeps
table counts, Li variants, layout stages and hardware scenarios separate. The cell tagged
`results-export` supplies the numbers used by the registry.

In [7]:
def registry_inputs(params):
    return {
        "hamiltonian": {
            "kind": "thc_cost_model", "orbitals": params["orbitals"],
            "electrons": params["electrons"], "thc_rank": params["M"],
            "lambda_value": params["lam"],
            "coefficient_bit_precision": params["chi"],
            "rotation_bit_precision": params["beta"],
        },
        "target_precision_hartree": PHASE_ERROR_HA,
        "trial_state": "not_prepared_in_analytic_resource_cost_model",
    }

def registry_logical(result):
    # Unmeasured gate classes are omitted rather than invented or set to null.
    return {key: value for key, value in result.items()
            if key != "step_toffolis" and value is not None}


In [8]:
results_record = {
    "schema_version": "1.0", "claim_id": CLAIM_ID, "reproduction_id": REPRODUCTION_ID,
    "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "producer": {"entrypoint": "notebook.ipynb", "cell_tag": "results-export"},
    "tooling": {"python": platform.python_version(), "packages": {
        name: metadata.version(name) for name in
        ("openfermion", "cirq-core", "qdk", "qdk-chemistry", "numpy", "scipy", "pandas", "jsonschema")}},
    "inputs": registry_inputs(CASES["reiher-m350"]),
    "logical": registry_logical(counts["reiher-m350"]),
    "physical": physical_scenarios,
    "logical_variants": [
        {"case_id": key, "inputs": registry_inputs(CASES[key]),
         "logical": registry_logical(counts[key])} for key in ("li-m350", "li-m450")],
    "layout": layout_logical,
}
with Path("results.json").open("w", encoding="utf-8", newline="\n") as handle:
    json.dump(results_record, handle, indent=2, allow_nan=False)
    handle.write("\n")
print("Wrote results.json from the calculated resources; validate with qacclaims validate.")


Wrote results.json from the calculated resources; validate with qacclaims validate.
